# **Transform Circuits Data**
  1. Read bronze circuits table   
  2. Keep only the columns required for analytics (Drop url column)
  3. Standardise Column using snake_case (circuitid -> circuit_id,circuitname -> circuit_name)
  4. Rename Columns to make them more meaningful ( lat -> latitude, long -> longitude)
  5. Filter out rows  where circuit_id is null (business key validation)
  6. Remove duplicate records 
  7. Transform values of columns circuit_name and locality to Title Case 
  8. Write transformated data to silver circuits table 
![](/Workspace/Users/uppalapatiususp@gmail.com/Pavan_Azure_DataEngineer_Projects/Formula1_Project/03-silver/DataFlow.png)

In [0]:
%run ../00-common/01_Environment-Config

In [0]:
from pyspark.sql.functions import * 
bronze_table_nm = f"{catalog_name}.{bronze_schema}.circuits"
silver_table_nm = f"{catalog_name}.{silver_schema}.circuits"

### Step 1 - Read bronze circuits table

In [0]:
#Below is one way of reading data from table 
circuits_df = (
    spark.table(bronze_table_nm)    
)

In [0]:
#Below is another way of reading data from table this one allows to add options 
circuits_df = (
    spark.read.table(bronze_table_nm)   
    
)

In [0]:
# display(df_circuits)

### Step 2 - Keep only the columns requried for analytics (Drop Url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
circuits_df.select(
    "circuitId",
    "circuitName",
    "lat",
    "long",
    "locality",
    "country",
    "ingestion_timestamp",
    "source_file"    
    )

In [0]:
circuits_select_df =  circuits_df.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingestion_timestamp"),
    F.col("source_file")    
    )

### Step 3,4 - Standardise Column Names
- Standardize columns names using snake_case (circuitid -> circuit_id, circuitname -> circuit_name)
- Rename columns to make them more meaningful (lat -> latitude,long -> longitude)

In [0]:
circuits_renamed_df = (
    circuits_select_df.withColumnRenamed("circuitId", "circuit_id")
    .withColumnRenamed("circuitName", "circuit_name")
    .withColumnRenamed("lat", "latitude")
    .withColumnRenamed("long", "longitude")
    .withColumnRenamed("locality", "locality")
    .withColumnRenamed("country", "country")
    .withColumnRenamed("ingestion_timestamp", "ingestion_timestamp")
    .withColumnRenamed("source_file", "source_file")
)

In [0]:
circuits_renamed_df = (
    circuits_select_df.withColumnsRenamed(
        {
            "circuitId": "circuit_id",
            "circuitName": "circuit_name",
            "lat": "latitude",
            "long": "longitude",
            "locality": "locality",
            "country": "country",
            "ingestion_timestamp": "ingestion_timestamp",
            "source_file": "source_file"
        })
)

In [0]:
# display(circuits_renamed_df)

### Step -5 Filter out rows where circuit id is null (business key validation)

In [0]:
# circuits_valid_df = circuits_renamed_df.filter("circuit_id IS NOT NULL")
# display(circuits_valid_df)


In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)
# display(circuits_valid_df)

### Step - 6 Remove Duplicates Records

In [0]:
# circuits_distinct_df = circuits_valid_df.distinct()
# display(circuits_distinct_df)

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])
# display(circuits_distinct_df)

### Step - 7 Transform values of columns circuit_name and locality to Title Case 

In [0]:
circuits_final_df = circuits_distinct_df.withColumns({
    "circuit_name": F.initcap(F.col("circuit_name")),
    "locality": F.initcap(F.col("locality"))
})
# display(circuits_final_df)



In [0]:
(
    circuits_final_df
        .write.mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table_nm)
)

In [0]:
spark.table(silver_table_nm).display()